# Coverage Prediction Model

Trains an XGBoost model on existing branch-tract coverage data to predict
what coverage a hypothetical new library branch would provide.

Given candidate coordinates, predicts the overlap with each census tract
and computes the delta on current coverage scores.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Pipeline Component

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc:8443"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "pandas", "pyarrow", "numpy", "scikit-learn", "xgboost", "shapely", "pyproj", "mlflow"],
    pip_index_urls=["https://pypi.org/simple"],
)
def train_and_predict(
    mlflow_tracking_uri: str,
    new_branch_lat: float,
    new_branch_lon: float,
    mode: str = "walking",
    travel_time: int = 10,
    predictions_out: Output[Dataset] = None,
):
    import io
    import os
    import math
    import logging
    import pickle
    import numpy as np
    import pandas as pd
    import boto3
    import mlflow
    from mlflow.models.signature import infer_signature
    from shapely import wkt
    from pyproj import Geod
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, r2_score
    from xgboost import XGBRegressor

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("coverage_model")

    geod = Geod(ellps="WGS84")

    # --- MLflow setup ---
    sa_token_path = "/var/run/secrets/kubernetes.io/serviceaccount/token"
    if os.path.exists(sa_token_path):
        with open(sa_token_path) as f:
            os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()
    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("cpl-coverage-model")
    logger.info(f"MLflow tracking URI: {mlflow_tracking_uri}")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # --- Helper functions ---
    def haversine_km(lat1, lon1, lat2, lon2):
        r = 6371.0
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
        return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    def bearing_deg(lat1, lon1, lat2, lon2):
        dlon = math.radians(lon2 - lon1)
        lat1, lat2 = math.radians(lat1), math.radians(lat2)
        x = math.sin(dlon) * math.cos(lat2)
        y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
        return (math.degrees(math.atan2(x, y)) + 360) % 360

    # --- Load data ---
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_details.parquet")
    coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(coverage)} coverage detail records")

    obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
    branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
    branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
    branch_coords = branches.set_index("BRANCH")[["LATITUDE", "LONGITUDE"]].to_dict("index")
    logger.info(f"Loaded {len(branches)} active branches")

    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(tracts)} census tracts")

    # Compute tract centroids and areas
    tract_info = {}
    for _, row in tracts.iterrows():
        geom = wkt.loads(row["geometry"])
        centroid = geom.centroid
        area, _ = geod.geometry_area_perimeter(geom)
        tract_info[row["geoid"]] = {
            "centroid_lat": centroid.y,
            "centroid_lon": centroid.x,
            "area_sqkm": abs(area) / 1e6,
            "total_population": row.get("total population", 0),
        }

    # --- Build training data ---
    modes_tt = coverage[["mode", "travel_time"]].drop_duplicates().values.tolist()
    all_geoids = set(tracts["geoid"])
    all_branches = set(coverage["branch"].unique())

    train_rows = []

    for m, tt in modes_tt:
        subset = coverage[(coverage["mode"] == m) & (coverage["travel_time"] == tt)]
        covered_pairs = set(zip(subset["branch"], subset["geoid"]))

        for _, row in subset.iterrows():
            b = row["branch"]
            g = row["geoid"]
            if b not in branch_coords or g not in tract_info:
                continue
            bc = branch_coords[b]
            ti = tract_info[g]
            dist = haversine_km(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
            bear = bearing_deg(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
            train_rows.append({
                "distance_km": dist,
                "bearing_sin": math.sin(math.radians(bear)),
                "bearing_cos": math.cos(math.radians(bear)),
                "tract_area_sqkm": ti["area_sqkm"],
                "travel_time": tt,
                "mode_driving": 1 if m == "driving" else 0,
                "mode_cycling": 1 if m == "cycling" else 0,
                "mode_walking": 1 if m == "walking" else 0,
                "overlap": row["overlap_proportion"],
            })

        for b in all_branches:
            if b not in branch_coords:
                continue
            bc = branch_coords[b]
            for g in all_geoids:
                if (b, g) in covered_pairs or g not in tract_info:
                    continue
                ti = tract_info[g]
                dist = haversine_km(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
                bear = bearing_deg(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
                train_rows.append({
                    "distance_km": dist,
                    "bearing_sin": math.sin(math.radians(bear)),
                    "bearing_cos": math.cos(math.radians(bear)),
                    "tract_area_sqkm": ti["area_sqkm"],
                    "travel_time": tt,
                    "mode_driving": 1 if m == "driving" else 0,
                    "mode_cycling": 1 if m == "cycling" else 0,
                    "mode_walking": 1 if m == "walking" else 0,
                    "overlap": 0.0,
                })

    train_df = pd.DataFrame(train_rows)
    logger.info(f"Training data: {len(train_df)} rows ({(train_df['overlap'] > 0).sum()} positive, {(train_df['overlap'] == 0).sum()} negative)")

    # --- Train model ---
    feature_cols = ["distance_km", "bearing_sin", "bearing_cos", "tract_area_sqkm",
                    "travel_time", "mode_driving", "mode_cycling", "mode_walking"]
    X = train_df[feature_cols]
    y = train_df["overlap"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="reg:squarederror",
        random_state=42,
    )

    with mlflow.start_run(run_name=f"coverage-{mode}-{travel_time}min") as run:
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        rmse = mean_squared_error(y_test, y_pred, squared=False)
        r2 = r2_score(y_test, y_pred)
        logger.info(f"Model performance — RMSE: {rmse:.4f}, R²: {r2:.4f}")

        mlflow.log_params({
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "mode": mode,
            "travel_time": travel_time,
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "positive_samples": int((train_df["overlap"] > 0).sum()),
            "negative_samples": int((train_df["overlap"] == 0).sum()),
            "new_branch_lat": new_branch_lat,
            "new_branch_lon": new_branch_lon,
        })

        mlflow.log_metrics({
            "rmse": rmse,
            "r2": r2,
        })

        importance = dict(zip(feature_cols, model.feature_importances_.tolist()))
        for feat, imp in importance.items():
            mlflow.log_metric(f"importance_{feat}", imp)

        signature = infer_signature(X_test, y_pred)
        mlflow.xgboost.log_model(
            model,
            artifact_path="coverage_model",
            signature=signature,
            registered_model_name="cpl-coverage-model",
        )
        logger.info(f"MLflow run ID: {run.info.run_id}")

    # Save model to S3 as well
    model_bytes = pickle.dumps(model)
    s3.put_object(Bucket=bucket, Key="models/coverage_model.pkl", Body=model_bytes)

    # --- Predict for new branch ---
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_scores.parquet")
    current_scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    current_scores = current_scores[(current_scores["mode"] == mode) & (current_scores["travel_time"] == travel_time)]
    current_map = current_scores.set_index("geoid")["percent_covered"].to_dict()

    pred_rows = []
    for geoid, ti in tract_info.items():
        dist = haversine_km(new_branch_lat, new_branch_lon, ti["centroid_lat"], ti["centroid_lon"])
        bear = bearing_deg(new_branch_lat, new_branch_lon, ti["centroid_lat"], ti["centroid_lon"])
        pred_rows.append({
            "geoid": geoid,
            "distance_km": dist,
            "bearing_sin": math.sin(math.radians(bear)),
            "bearing_cos": math.cos(math.radians(bear)),
            "tract_area_sqkm": ti["area_sqkm"],
            "travel_time": travel_time,
            "mode_driving": 1 if mode == "driving" else 0,
            "mode_cycling": 1 if mode == "cycling" else 0,
            "mode_walking": 1 if mode == "walking" else 0,
            "total_population": ti["total_population"],
        })

    pred_df = pd.DataFrame(pred_rows)
    pred_df["predicted_overlap"] = model.predict(pred_df[feature_cols]).clip(0, 1)

    pred_df["current_coverage"] = pred_df["geoid"].map(current_map).fillna(0)
    pred_df["new_coverage"] = (pred_df["current_coverage"] + pred_df["predicted_overlap"]).clip(0, 1)
    pred_df["delta"] = pred_df["new_coverage"] - pred_df["current_coverage"]

    result = pred_df[["geoid", "total_population", "current_coverage", "predicted_overlap",
                       "new_coverage", "delta", "distance_km"]].copy()
    result = result.sort_values("delta", ascending=False).reset_index(drop=True)

    # Write predictions to S3
    buf = io.BytesIO()
    result.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_predictions.parquet", Body=buf.getvalue())

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_predictions.csv", Body=buf.getvalue())

    result.to_parquet(predictions_out.path, index=False)

    affected = result[result["delta"] > 0.01]
    pop_affected = affected["total_population"].astype(float).sum()
    logger.info(
        f"Predicted coverage for new branch at ({new_branch_lat}, {new_branch_lon}) [{mode}/{travel_time}min]:\n"
        f"  Tracts with >1% coverage gain: {len(affected)}\n"
        f"  Population in affected tracts: {pop_affected:,.0f}\n"
        f"  Max delta: {result['delta'].max():.3f}\n"
        f"  Mean delta (affected): {affected['delta'].mean():.3f}"
    )

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-coverage-model",
    description="Train coverage model and predict impact of a new library branch",
)
def coverage_model_pipeline(
    new_branch_lat: float = 41.8781,
    new_branch_lon: float = -87.6298,
    mode: str = "walking",
    travel_time: int = 10,
):
    task = train_and_predict(
        mlflow_tracking_uri=MLFLOW_TRACKING_URI,
        new_branch_lat=new_branch_lat,
        new_branch_lon=new_branch_lon,
        mode=mode,
        travel_time=travel_time,
    )
    kubernetes.use_secret_as_env(
        task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

## Submit

In [ ]:
import subprocess
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(coverage_model_pipeline, "coverage_model_pipeline.yaml")

try:
    client.upload_pipeline("coverage_model_pipeline.yaml", pipeline_name="cpl-coverage-model")
    print("Pipeline definition registered.")
except ApiException as e:
    if e.status == 409:
        print("Pipeline definition already exists.")
    else:
        raise

run = client.create_run_from_pipeline_func(
    coverage_model_pipeline,
    experiment_name="cpl-coverage",
    run_name="coverage-model-run",
    enable_caching=False,
)
print(f"Pipeline run submitted: {run.run_id}")